# DeiT-LT Exact Reproduction (Kaggle)

This notebook reproduces the exact methodology from the CVPR 2024 paper **DeiT-LT: Distillation strikes back for Vision Transformer training on Long-Tailed datasets**.

- **Model**: DeiT-Tiny (5M Params)
- **Teacher**: PaCo SAM ResNet-32
- **Dataset**: CIFAR-10 LT (Imbalance Factor 50)
- **Hardware Target**: Kaggle 2x T4 GPUs (or single GPU)


## 1. Environment Setup


In [ ]:
!pip install timm==0.4.12 torch torchvision


## 2. Imports and Teacher Weights


In [ ]:
import os
import time
import math
import urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy
from timm.scheduler import create_scheduler
from timm.optim import create_optimizer

# 1. Download Teacher Weights (IF 50)
teacher_url = "https://api.wandb.ai/artifactsV2/default/pradipto611/QXJ0aWZhY3Q6Nzk3NzA4NTEx/fc15814c0ce158e6987110b256248e18/paco_sam_ckpt_cf10_if50.pth.tar"
weight_path = "teacher_cifar10_lt_if50.pth"
if not os.path.exists(weight_path):
    print("Downloading Official ResNet-32 Teacher (IF=50)...")
    urllib.request.urlretrieve(teacher_url, weight_path)
    print("Downloaded!")



## 3. Dataset Generation (CIFAR-10 LT)


In [ ]:
# Exact Pareto distribution extraction used in the official paper
def make_long_tail(dataset, num_classes=10, imb_factor=0.02): # 0.02 = IF 50
    img_max = len(dataset.samples) / num_classes
    img_num_per_cls = []
    for cls_idx in range(num_classes):
        num = img_max * (imb_factor ** (cls_idx / (num_classes - 1.0)))
        img_num_per_cls.append(int(num))
        
    new_samples = []
    targets_np = np.array(dataset.targets)
    for the_class in range(num_classes):
        idx = np.where(targets_np == the_class)[0]
        np.random.shuffle(idx)
        selec_idx = idx[:img_num_per_cls[the_class]]
        for i in selec_idx:
            new_samples.append(dataset.samples[i])
            
    dataset.samples = new_samples
    dataset.targets = [s[1] for s in new_samples]
    return dataset, img_num_per_cls



## 4. Models (DeiT-Tiny Dual-Expert & ResNet-32 Teacher)


In [ ]:
# --- TEACHER: ResNet-32 ---
import torch.nn.init as init
from torch.nn import Parameter

class LambdaLayer(nn.Module):
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd
    def forward(self, x):
        return self.lambd(x)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = LambdaLayer(lambda x:
                                        F.pad(x[:, :, ::2, ::2], (0, 0, 0, 0, planes//4, planes//4), "constant", 0))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet_s(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet_s, self).__init__()
        self.in_planes = 16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.linear = nn.Linear(64, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, out.size()[3])
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

def resnet32():
    return ResNet_s(BasicBlock, [5, 5, 5])

# --- STUDENT: DeiT-Tiny (Clean-room Implementation) ---
from timm.models.vision_transformer import Block

class DeiTLT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, embed_dim=192, depth=12, num_heads=3, num_classes=10):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 2, embed_dim))
        
        self.blocks = nn.ModuleList([Block(embed_dim, num_heads, mlp_ratio=4, qkv_bias=True) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        
        self.head_cls = nn.Linear(embed_dim, num_classes)
        self.head_dist = nn.Linear(embed_dim, num_classes)
        
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.dist_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
            
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        dist_tokens = self.dist_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, dist_tokens, x), dim=1)
        x = x + self.pos_embed
        
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        
        logits_cls = self.head_cls(x[:, 0])
        logits_dist = self.head_dist(x[:, 1])
        return logits_cls, logits_dist



## 5. Main Training Pipeline


In [ ]:
# --- Configuration ---
class Config:
    epochs = 300 # Tiny model needs fewer epochs than Base, but you can scale this
    batch_size = 128
    lr = 5e-4
    warmup_epochs = 5
    drw_epoch = int(epochs * 0.9) # DRW in last 10%
    weight_decay = 0.05
    mixup = 0.8
    cutmix = 1.0
    imb_factor = 0.02 # IF 50
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

args = Config()

# --- Dataloaders ---
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
# Note: In Kaggle, use torchvision.datasets.CIFAR10 directly and extract it properly, 
# or use Kaggle Datasets for faster loading. For simplicity here:
cifar_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
cifar_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Convert CIFAR10 to ImageFolder-like format for our make_long_tail function
class CIFARWrapper:
    def __init__(self, ds):
        self.samples = [(img, target) for img, target in zip(ds.data, ds.targets)]
        self.targets = ds.targets
wrapped_train = CIFARWrapper(cifar_train)
lt_train, cls_distribution = make_long_tail(wrapped_train, imb_factor=args.imb_factor)
cifar_train.data = np.stack([s[0] for s in lt_train.samples])
cifar_train.targets = lt_train.targets

train_loader = DataLoader(cifar_train, batch_size=args.batch_size, shuffle=True, num_workers=2, drop_last=True)
test_loader = DataLoader(cifar_test, batch_size=args.batch_size, shuffle=False, num_workers=2)

print("Class distribution:", cls_distribution)

# --- Per Class Weights (for DRW) ---
beta = 0.9999
effective_num = 1.0 - np.power(beta, cls_distribution)
per_cls_weights = (1.0 - beta) / np.array(effective_num)
per_cls_weights = per_cls_weights / np.sum(per_cls_weights) * 10
per_cls_weights = torch.FloatTensor(per_cls_weights).to(args.device)

# --- Mixup ---
mixup_fn = Mixup(mixup_alpha=args.mixup, cutmix_alpha=args.cutmix, label_smoothing=0.1, num_classes=10)

# --- Teacher Load ---
teacher = resnet32()
ckpt = torch.load("teacher_cifar10_lt_if50.pth", map_location='cpu')
new_state_dict = {}
for k, v in ckpt['state_dict'].items():
    k = k.replace('module.', '')
    if k.startswith('encoder_q.'):
        if 'fc.' in k: continue
        new_state_dict[k.replace('encoder_q.', '')] = v
    elif k.startswith('linear.'):
        new_state_dict[k] = v
teacher.load_state_dict(new_state_dict, strict=False)
teacher = teacher.to(args.device)
teacher.eval()
for p in teacher.parameters(): p.requires_grad = False

# --- Student Model & Optimizer ---
model = DeiTLT().to(args.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
scaler = GradScaler()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
base_criterion = SoftTargetCrossEntropy()

# --- Distillation Loss ---
def distillation_loss(student_logits, teacher_logits, temperature=3.0, hard=True):
    if hard:
        targets = teacher_logits.argmax(dim=1)
        return F.cross_entropy(student_logits, targets)
    else:
        log_prob_s = F.log_softmax(student_logits / temperature, dim=1)
        prob_t = F.softmax(teacher_logits / temperature, dim=1)
        return F.kl_div(log_prob_s, prob_t, reduction='batchmean') * (temperature ** 2)

# --- Training Loop ---
for epoch in range(args.epochs):
    model.train()
    total_loss, total_cls, total_dist = 0, 0, 0
    
    # DRW Logic
    use_drw = epoch >= args.drw_epoch
    weight = 2.0 if use_drw else 1.0
    
    for i, (images, targets) in enumerate(train_loader):
        images, targets = images.to(args.device), targets.to(args.device)
        
        # Disable Mixup during DRW phase (as per paper)
        if not use_drw:
            images, targets = mixup_fn(images, targets)
        else:
            targets = torch.nn.functional.one_hot(targets, num_classes=10).float()
            
        with autocast():
            with torch.no_grad():
                # Teacher takes 32x32 for CIFAR
                images_32 = F.interpolate(images, size=(32, 32), mode='bilinear', align_corners=False)
                teacher_logits = teacher(images_32)
                
            logits_cls, logits_dist = model(images)
            
            if use_drw:
                loss_cls = F.cross_entropy(logits_cls, targets.argmax(dim=1), weight=per_cls_weights)
            else:
                loss_cls = base_criterion(logits_cls, targets)
                
            loss_dist = distillation_loss(logits_dist, teacher_logits, hard=True)
            loss = loss_cls + (weight * loss_dist)
            
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
    scheduler.step()
    
    if (epoch + 1) % 10 == 0:
        # Evaluate
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(args.device), lbls.to(args.device)
                l_cls, l_dist = model(imgs)
                l_avg = (l_cls + l_dist) / 2
                preds = l_avg.argmax(dim=1)
                correct += (preds == lbls).sum().item()
                total += lbls.size(0)
        acc = correct / total * 100
        print(f"Epoch {epoch+1}/{args.epochs} | Loss: {total_loss/len(train_loader):.3f} | Test Acc: {acc:.2f}%")

